In [1]:
!pip install pandas


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install numpy


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install matplotlib


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip install scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
df = pd.read_csv('german_credit_data.csv')

In [6]:
df.head()

,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
0,0,67,male,2,own,NaN,little,1169,6,radio/TV
1,1,22,female,2,own,little,moderate,5951,48,radio/TV
2,2,49,male,1,own,little,NaN,2096,12,education
3,3,45,male,2,free,little,little,7882,42,furniture/equipment
4,4,53,male,2,free,little,little,4870,24,car


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Unnamed: 0        1000 non-null   int64
 1   Age               1000 non-null   int64
 2   Sex               1000 non-null   str  
 3   Job               1000 non-null   int64
 4   Housing           1000 non-null   str  
 5   Saving accounts   817 non-null    str  
 6   Checking account  606 non-null    str  
 7   Credit amount     1000 non-null   int64
 8   Duration          1000 non-null   int64
 9   Purpose           1000 non-null   str  
dtypes: int64(5), str(5)
memory usage: 78.3 KB


In [8]:
df.isnull().sum()

Unnamed: 0            0
Age                   0
Sex                   0
Job                   0
Housing               0
Saving accounts     183
Checking account    394
Credit amount         0
Duration              0
Purpose               0
dtype: int64

In [9]:
# there is 183 savings accounts with  missing values
# there is 294 checking accounts with missing values

In [9]:
print(df.columns)

Index(['Unnamed: 0', 'Age', 'Sex', 'Job', 'Housing', 'Saving accounts',
       'Checking account', 'Credit amount', 'Duration', 'Purpose'],
      dtype='str')


In [10]:
# we should drop the column "Unnamed : 0"
df = df.drop(columns = ['Unnamed: 0'])

print(df.columns)

Index(['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account',
       'Credit amount', 'Duration', 'Purpose'],
      dtype='str')


In [11]:
# filling the missing values with 'unknown'
df['Saving accounts'] = df['Saving accounts'].fillna('unknown')
df['Checking account'] = df['Checking account'].fillna('unknown')

In [ ]:
#create a binary target variable based on Credit amount median
median_credit = df['Credit amount'].median()
df['High_Credit'] = (df['Credit amount']> median_credit).astype(int)

In [13]:
# define the features and the target variables
X = df.drop(columns = ['Credit amount', 'High_Credit'])
y = df['High_Credit']

In [14]:
print(df.columns)

Index(['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account',
       'Credit amount', 'Duration', 'Purpose', 'High_Credit'],
      dtype='str')


In [15]:
from sklearn.model_selection import train_test_split as tts

X_train, X_test, y_train, y_test = tts(X, y, test_size=0.2, random_state=42)

In [20]:
numerical = ['Age', 'Duration']
categorical = ['Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']

In [21]:
X_train_numeric = X_train[numerical]
X_train_categoric = X_train[categorical]

In [30]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
scaler = StandardScaler()
X_train_numeric_scaled = scaler.fit_transform(X_train_numeric)
X_train_numeric_scaled = pd.DataFrame(X_train_numeric_scaled, columns = numerical)

encoder = OneHotEncoder()
X_train_categoric_encoded = encoder.fit_transform(X_train_categoric)
X_train_categoric_encoded = pd.DataFrame(X_train_categoric_encoded.toarray(), columns = encoder.get_feature_names_out())

X_train_scaled_encoded = pd.concat([X_train_numeric_scaled, X_train_categoric_encoded], axis = 1)

In [34]:
X_test_numeric = X_test[numerical]
X_test_categoric = X_test[categorical]

X_test_numeric_scaled = scaler.transform(X_test_numeric)
X_test_numeric_scaled = pd.DataFrame(X_test_numeric_scaled, columns = numerical)

X_test_categoric_encoded = encoder.transform(X_test_categoric)
X_test_categoric_encoded  = pd.DataFrame(X_test_categoric_encoded.toarray(), columns = encoder.get_feature_names_out())

X_test_scaled_encoded = pd.concat([X_test_numeric_scaled, X_test_categoric_encoded], axis = 1)


In [35]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state = 1)
model.fit(X_train_scaled_encoded, y_train)

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_auc_score

y_pred = model.predict(X_test_scaled_encoded)

print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(precision_score(y_test, y_pred))
print(recall_score(y_test, y_pred))
print(f1_score(y_test, y_pred))
print(roc_auc_score(y_test, y_pred))


[[84 22]
 [29 65]]
0.745
              precision    recall  f1-score   support

           0       0.74      0.79      0.77       106
           1       0.75      0.69      0.72        94

    accuracy                           0.74       200
   macro avg       0.75      0.74      0.74       200
weighted avg       0.75      0.74      0.74       200

0.7471264367816092
0.6914893617021277
0.7182320441988951
0.7419710959454034


In [37]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_scaled_encoded, y_train)

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_auc_score

y_pred = model.predict(X_test_scaled_encoded)

print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(precision_score(y_test, y_pred))
print(recall_score(y_test, y_pred))
print(f1_score(y_test, y_pred))
print(roc_auc_score(y_test, y_pred))


[[87 19]
 [26 68]]
0.775
              precision    recall  f1-score   support

           0       0.77      0.82      0.79       106
           1       0.78      0.72      0.75        94

    accuracy                           0.78       200
   macro avg       0.78      0.77      0.77       200
weighted avg       0.78      0.78      0.77       200

0.7816091954022989
0.723404255319149
0.7513812154696132
0.7720794861501405


In [38]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled_encoded, y_train)

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_auc_score

y_pred = model.predict(X_test_scaled_encoded)

print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(precision_score(y_test, y_pred))
print(recall_score(y_test, y_pred))
print(f1_score(y_test, y_pred))
print(roc_auc_score(y_test, y_pred))


[[74 32]
 [29 65]]
0.695
              precision    recall  f1-score   support

           0       0.72      0.70      0.71       106
           1       0.67      0.69      0.68        94

    accuracy                           0.69       200
   macro avg       0.69      0.69      0.69       200
weighted avg       0.70      0.69      0.70       200

0.6701030927835051
0.6914893617021277
0.680628272251309
0.6948012846246487
